In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import glob
import os

SCAN_FILES = sorted(glob.glob("out/scan*_stripped.csv.gz"))
CACHE = "out/combined.parquet"

# lfs quota totals (852.6 TiB = 852.6 * 1024^4 bytes)
QUOTA_TOTAL_TB = 852.6 * (1024**4) / 1e12
QUOTA_TOTAL_FILES = 132_818_236

if os.path.exists(CACHE):
    print("Loading from parquet cache...")
    df = pd.read_parquet(CACHE)
else:
    print(f"Loading {len(SCAN_FILES)} csv.gz files...")
    frames = []
    for f in SCAN_FILES:
        frames.append(pd.read_csv(f, keep_default_na=False, dtype={"size_bytes": str, "uid": str}))
    df = pd.concat(frames, ignore_index=True)

    # vectorized: first path component as user identifier
    df["user"] = df["path"].str.split("/").str[1].fillna("unknown")

    df_files = df[df["error"] == ""].copy()
    df_files["size_bytes"] = df_files["size_bytes"].astype(int)
    df_files["size_tb"] = df_files["size_bytes"] / 1e12
    df_files["mtime"] = pd.to_datetime(df_files["mtime"])

    # vectorized directory extraction (first 3 components after split)
    parts = df_files["path"].str.split("/")
    df_files["directory"] = parts.apply(lambda p: "/".join(p[:4]))

    df_errors = df[df["error"] != ""][["path", "user", "error"]].copy()

    df = {"files": df_files, "errors": df_errors}
    pd.concat([df_files, df_errors]).to_parquet(CACHE)
    print("Saved parquet cache.")

if isinstance(df, dict):
    files = df["files"]
    errors = df["errors"]
else:
    files = df[df["error"] == ""].copy()
    errors = df[df["error"] != ""].copy()

denied_users = set(errors["user"].unique())

scanned_tb = files["size_tb"].sum()
scanned_files = len(files)
unaccounted_tb = QUOTA_TOTAL_TB - scanned_tb
unaccounted_files = QUOTA_TOTAL_FILES - scanned_files

print(f"Total files (scanned):  {scanned_files:,}")
print(f"Total files (quota):    {QUOTA_TOTAL_FILES:,}")
print(f"Unaccounted files:      {unaccounted_files:,}")
print(f"Total size (scanned):   {scanned_tb:.2f} TB")
print(f"Total size (quota):     {QUOTA_TOTAL_TB:.2f} TB")
print(f"Unaccounted size:       {unaccounted_tb:.2f} TB")

In [ ]:
# Pie charts: by volume and by file count, per user
def label(user):
    return user + ("*" if user in denied_users else "")

by_volume = files.groupby("user")["size_tb"].sum().sort_values(ascending=False)
by_count  = files.groupby("user")["size_bytes"].count().sort_values(ascending=False)

# add unaccounted slice
by_volume["unaccounted"] = unaccounted_tb
by_count["unaccounted"]  = unaccounted_files

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, data, title, fmt in [
    (axes[0], by_volume, "Disk usage by volume (TB)", lambda v: f"{v:.1f} TB"),
    (axes[1], by_count,  "Disk usage by file count",  lambda v: f"{v/1e6:.1f}M"),
]:
    labels = [label(u) if u != "unaccounted" else "unaccounted" for u in data.index]
    colors = ["#cccccc" if u == "unaccounted" else None for u in data.index]
    wedges, texts, autotexts = ax.pie(
        data.values,
        labels=labels,
        colors=colors,
        autopct=lambda p: fmt(p / 100 * data.sum()),
        pctdistance=0.75,
        startangle=90,
    )
    for t in autotexts:
        t.set_fontsize(8)
    ax.set_title(title)

fig.suptitle("* = has permission-denied entries  |  grey = unaccounted (permission denied)", fontsize=10, y=0.02)
fig.tight_layout()
plt.show()

In [ ]:
# Top 20 largest directories
by_dir = (
    files.groupby("directory")["size_tb"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(10, 6))
by_dir.plot.barh(ax=ax)
ax.invert_yaxis()
ax.set_xlabel("Total size (TB)")
ax.set_title("Top 20 largest directories")
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:.1f}"))
fig.tight_layout()
plt.show()

In [ ]:
# Per-user statistics table
stats = files.groupby("user").agg(
    total_files=("size_bytes", "count"),
    total_tb=("size_tb", "sum"),
    avg_file_size_mb=("size_bytes", lambda x: x.mean() / 1e6),
    median_file_size_mb=("size_bytes", lambda x: x.median() / 1e6),
    largest_file_gb=("size_bytes", lambda x: x.max() / 1e9),
).sort_values("total_tb", ascending=False)

stats.index = [label(u) for u in stats.index]
stats.columns = ["Total files", "Total (TB)", "Avg file size (MB)", "Median file size (MB)", "Largest file (GB)"]
stats["Total files"] = stats["Total files"].apply(lambda x: f"{x:,}")
stats["Total (TB)"] = stats["Total (TB)"].apply(lambda x: f"{x:.2f}")
stats["Avg file size (MB)"] = stats["Avg file size (MB)"].apply(lambda x: f"{x:.1f}")
stats["Median file size (MB)"] = stats["Median file size (MB)"].apply(lambda x: f"{x:.1f}")
stats["Largest file (GB)"] = stats["Largest file (GB)"].apply(lambda x: f"{x:.1f}")

stats